In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "/kaggle/input/kunuz-cleaned/kunuz_final.csv"

# Load the latest version
df = pd.read_csv(file_path)

df = df[df['gender'] != 'unknown']
# not 17620 
len(df)

16154

In [ ]:
counts = df["gender"].value_counts()
counts2 = df["year"].value_counts()
counts3 = df["category"].value_counts()
print(f'gender: {counts} year: {counts2}, category: {counts3}')

In [ ]:
import os, random, math, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

# -----------------
# Config
# -----------------
SEED = 42
BATCH_TRAIN = 32
BATCH_VAL = 32
EPOCHS = 25
LR = 2e-5
MAX_LEN = 512
MODEL_NAME = "elmurod1202/bertbek-news-big-cased"

device = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

In [1]:
from transformers import DataCollatorWithPadding
from sklearn.preprocessing import LabelEncoder

le_gender = LabelEncoder()
le_age = LabelEncoder()
le_industry = LabelEncoder()

df["gender_lbl"]   = le_gender.fit_transform(df["gender"])
df["year_lbl"] = le_age.fit_transform(df["year"])
df["topic_lbl"]  = le_industry.fit_transform(df["category"])


# -------- Dataset (no fixed padding here) --------
class MTDataset(Dataset):
    def __init__(self, df, max_len=MAX_LEN):
        self.texts = df["body"].astype(str).tolist()
        self.g = df["gender_lbl"].astype(int).tolist()
        self.y = df["year_lbl"].astype(int).tolist()
        self.t = df["topic_lbl"].astype(int).tolist()
        self.max_len = MAX_LEN
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = tok(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,   # cap length
            return_tensors="pt"        # <-- no padding="max_length"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["g"] = torch.tensor(self.g[idx], dtype=torch.float32)
        item["y"] = torch.tensor(self.y[idx], dtype=torch.long)
        item["t"] = torch.tensor(self.t[idx], dtype=torch.long)
        return item

# -------- Dynamic pad collator --------
collate_fn = DataCollatorWithPadding(tokenizer=tok, return_tensors="pt")

# -------- Split (keep your label encoders as you wrote) --------
train_df, val_df = train_test_split(
    df, test_size=0.15, random_state=SEED, stratify=df["gender_lbl"]
)

ds_tr, ds_va = MTDataset(train_df, max_len=MAX_LEN), MTDataset(val_df, max_len=MAX_LEN)

# Use workers + pin_memory for speed
dl_tr = DataLoader(
    ds_tr, batch_size=BATCH_TRAIN, shuffle=True,
    collate_fn=collate_fn, num_workers=4, pin_memory=True, persistent_workers=True
)
dl_va = DataLoader(
    ds_va, batch_size=BATCH_VAL, shuffle=False,
    collate_fn=collate_fn, num_workers=4, pin_memory=True, persistent_workers=True
)


# 1) counts (MUST come first)
year_counts  = df["year_lbl"].value_counts().sort_index()
topic_counts = df["topic_lbl"].value_counts().sort_index()

# 2) number of classes (optional; can also use year_counts.size)
num_year  = df["year_lbl"].nunique()
num_topic = df["topic_lbl"].nunique()

# 3) weights
year_w = year_counts.sum() / (num_year * year_counts)
year_w = torch.tensor(year_w.values, dtype=torch.float)

topic_w = topic_counts.sum() / (num_topic * topic_counts)
topic_w = torch.tensor(topic_w.values, dtype=torch.float)
topic_w = torch.clamp(topic_w, max=10.0)


gender_counts = df["gender_lbl"].value_counts().sort_index()
n_neg = gender_counts[0]   # label 0 (e.g. male)
n_pos = gender_counts[1]   # label 1 (e.g. female)
gender_pos_w = torch.tensor([n_neg / n_pos], dtype=torch.float)

gender_pos_w = gender_pos_w.to(device)
year_w  = year_w.to(device)
topic_w = topic_w.to(device)

print('hi')

2026-01-13 08:21:49.476709: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768292509.899677      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768292510.034792      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

NameError: name 'df' is not defined

In [ ]:
print(year_counts.index.tolist())
print(topic_counts.index.tolist())

In [ ]:
class MTLModel(nn.Module):
    def __init__(self, model_name, num_year, num_topic, year_w=None, topic_w=None, gender_pos_w=None):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hid = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)

        # projections
        self.proj_g = nn.Linear(hid, hid)
        self.proj_y = nn.Linear(hid, hid)
        self.proj_t = nn.Linear(hid, hid)

        # cross-attn blocks (NEW: 2 blocks)
        self.ca_y = nn.MultiheadAttention(embed_dim=hid, num_heads=4, batch_first=True)
        self.ca_t = nn.MultiheadAttention(embed_dim=hid, num_heads=4, batch_first=True)

        # heads
        self.head_g = nn.Linear(hid, 1)
        self.head_y = nn.Linear(hid, num_year)
        self.head_t = nn.Linear(hid, num_topic)

        # losses
        self.bce  = nn.BCEWithLogitsLoss(pos_weight=gender_pos_w)
        self.ce_y = nn.CrossEntropyLoss(weight=year_w,  label_smoothing=0.05)
        self.ce_t = nn.CrossEntropyLoss(weight=topic_w, label_smoothing=0.05)

        # uncertainty weights
        self.log_sigma_gender = nn.Parameter(torch.zeros(1))
        self.log_sigma_year   = nn.Parameter(torch.zeros(1))
        self.log_sigma_topic  = nn.Parameter(torch.zeros(1))

    def forward(self, input_ids, attention_mask, g=None, y=None, t=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = self.dropout(out.last_hidden_state[:, 0])  # CLS (B,H)

        # base task reps
        h_g = self.proj_g(h)
        h_y = self.proj_y(h)
        h_t = self.proj_t(h)

        # ---- Year attends to Gender (G -> Y) ----
        qy = h_y.unsqueeze(1)              # (B,1,H)
        kv_g = h_g.unsqueeze(1)            # (B,1,H)
        h_y_ca, _ = self.ca_y(qy, kv_g, kv_g)
        h_y_ca = h_y_ca.squeeze(1)         # (B,H)

        # ---- Topic attends to Gender (G -> T) ----
        qt = h_t.unsqueeze(1)              # (B,1,H)
        h_t_ca, _ = self.ca_t(qt, kv_g, kv_g)
        h_t_ca = h_t_ca.squeeze(1)         # (B,H)

        # logits
        lg = self.head_g(h_g).squeeze(1)
        ly = self.head_y(h_y_ca)
        lt = self.head_t(h_t_ca)

        losses = None
        if g is not None:
            Lg = self.bce(lg, g.float())
            Ly = self.ce_y(ly, y.long())
            Lt = self.ce_t(lt, t.long())
            loss = (
                torch.exp(-self.log_sigma_gender) * Lg + self.log_sigma_gender +
                torch.exp(-self.log_sigma_year)   * Ly + self.log_sigma_year +
                torch.exp(-self.log_sigma_topic)  * Lt + self.log_sigma_topic
            )
            losses = (loss, Lg.detach(), Ly.detach(), Lt.detach())

        return (lg, ly, lt), losses


model = MTLModel(
    MODEL_NAME,
    num_year,
    num_topic,
    year_w=year_w,
    topic_w=topic_w,
    gender_pos_w=gender_pos_w
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
num_train_steps = EPOCHS * len(dl_tr)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=num_train_steps)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

def evaluate():
    model.eval()
    g_probs, g_true, y_pred, y_true, t_pred, t_true = [], [], [], [], [], []
    with torch.no_grad():
        for batch in dl_va:
            for k in ("input_ids","attention_mask"): batch[k] = batch[k].to(device)
            g, y, t = batch["g"].to(device), batch["y"].to(device), batch["t"].to(device)
            (lg, ly, lt), _ = model(batch["input_ids"], batch["attention_mask"])
            g_probs.extend(torch.sigmoid(lg).cpu().numpy())
            g_true.extend(g.cpu().numpy())
            y_pred.extend(ly.argmax(1).cpu().numpy()); y_true.extend(y.cpu().numpy())
            t_pred.extend(lt.argmax(1).cpu().numpy()); t_true.extend(t.cpu().numpy())
    g_probs = np.array(g_probs); g_true = np.array(g_true)
    g_pred = (g_probs >= 0.5).astype(int)
    return {
        "G_acc": accuracy_score(g_true, g_pred),
        "G_f1": f1_score(g_true, g_pred, average="macro"),
        "Y_acc": accuracy_score(y_true, y_pred),
        "Y_f1": f1_score(y_true, y_pred, average="macro"),
        "T_acc": accuracy_score(t_true, t_pred),
        "T_f1": f1_score(t_true, t_pred, average="macro"),
    }

In [ ]:
import os
import numpy as np
from tqdm.auto import tqdm

# where to save on Kaggle
CKPT_DIR = "/kaggle/working/bertbek_mtl"
os.makedirs(CKPT_DIR, exist_ok=True)

for ep in range(1, EPOCHS + 1):
    model.train()
    tr_losses = []
    pbar = tqdm(dl_tr, desc=f"Epoch {ep}", leave=False)

    for batch in pbar:
        for k in ("input_ids", "attention_mask"):
            batch[k] = batch[k].to(device)

        g = batch["g"].to(device)
        y = batch["y"].to(device)
        t = batch["t"].to(device)

        # forward
        (_, _, _), losses = model(
            batch["input_ids"],
            batch["attention_mask"],
            g, y, t
        )

        loss = losses[0]
        if loss.dim() > 0:      # e.g., DataParallel -> [num_devices]
            loss = loss.mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        tr_losses.append(loss.item())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    # evaluation
    metrics = evaluate()
    comp = (metrics["G_f1"] + metrics["Y_f1"] + metrics["T_f1"]) / 3.0

    print(
        f"Epoch {ep:02d} | train {np.mean(tr_losses):.4f} | "
        f"G {metrics['G_acc']:.3f}/{metrics['G_f1']:.3f} | "
        f"Y {metrics['Y_acc']:.3f}/{metrics['Y_f1']:.3f} | "
        f"T {metrics['T_acc']:.3f}/{metrics['T_f1']:.3f} | comp {comp:.3f}"
    )
    print(torch.exp(-model.module.log_sigma_gender).item())
    print(torch.exp(-model.module.log_sigma_year).item())
    print(torch.exp(-model.module.log_sigma_topic).item())



    # save checkpoint for this epoch
    ckpt_path = os.path.join(CKPT_DIR, f"bertbek_mtl_epoch{ep}.pt")
    torch.save(model.state_dict(), ckpt_path)
    print(f"Saved checkpoint: {ckpt_path}")